In [1]:
# Parameters
MNIST_ROOT = "MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist"


# Argumentation-11 — Protocole de Debate & Securite

**Navigation** : [README](./README.md) | [<< 10-Privacy-Export](./Argumentation-10-Privacy-Export-Python.ipynb)

> Suite de la serie Argumentation : le **debate** comme mecanisme de securite
> (Christiano, Irving & Amodei, *AI Safety via Debate*, arXiv:1805.00899),
> rendu **execute et mesurable** sur CPU — un juge CNN sur MNIST arbitre un
> jeu de debat ou les agents revelent les pixels qu'ils invoquent.

Ce carnet implemente, de facon **deterministe et reproductible** :

1. le **jeu de debat** du §2 du paper (protocole de revelation) ;
2. la **reproduction MNIST §3.1** — un juge sur masques epars (4/6 pixels),
   avec le **gain marginal** que nous mesurons ici et l'**ecart au saut** que
   le paper rapporte (assumé, §3.1) ;
3. la **formalisation** du transcript de debat vers un **cadre d'argumentation
   Dung/ADF** (raccorde aux carnets 03 / 03d de la serie) ;
4. les **modes de defaillance** — mind-hack (§5.8, discussion declaree) et
   collusion (§5.10, demo mesuree) ;
5. **trois exercices** (stubs conformes, a completer).

> **Honnêteté du dispositif.** Il n'y a **pas de LLM texte** sur ce siege. Les
> « agents » sont donc des **politiques deterministes** (saliencer du juge),
> pas des modeles de langage. On modele la **structure** du protocole de la
> revelation comme *gage* qui structure l'evidence — pas le reglage LLM complet
> du paper. Les parties a texte libre (mind-hack) sont une **discussion
> declaree**, jamais une sortie fabriquée.


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 20261116
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(2)

# Racine MNIST : parametre papermill (defaut = chemin relatif au repo, gitignore
# via ArgumentAnalysis/output/). A l'execution, la lane injecte le chemin absolu.
# papermill-inject
# MNIST_ROOT "MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist"
os.makedirs(MNIST_ROOT, exist_ok=True)

print("torch", torch.__version__, "| threads", torch.get_num_threads(), "| MNIST_ROOT", MNIST_ROOT)

torch 2.14.1+cu126 | threads 2 | MNIST_ROOT MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist


### Lecture du résultat

Trois lignes de sortie, trois décisions.

La version de torch fixe l'environnement numérique qui produit tous les chiffres de ce carnet : `torch 2.14.1+cu126`. Le nombre de fils d'exécution (`threads 2`) fixe l'ordre dans lequel les multiplications matricielles s'additionnent, donc le dernier chiffre de l'accuracy — c'est la raison pour laquelle ce carnet rejoue à l'identique sur la même machine et dérive à la millième sur une autre. Le `MNIST_ROOT` est le seul point d'entrée paramétrable : il pointe vers un répertoire hors du dépôt (`ArgumentAnalysis/output/mnist`), ce qui évite d'écrire des données dans l'arbre versionné et rend la cellule idempotente — la relancer ne fait que réutiliser le cache déjà téléchargé.

Les deux graines (`20261116` pour torch, la même pour numpy) sont posées **avant** tout tirage aléatoire. C'est la condition qui rend l'entraînement du juge reproductible : sans elles, les trois epochs produiraient à chaque exécution un modèle différent, et la mesure du débat — qui repose entièrement sur ce juge — deviendrait irreproductible. Tout le reste du carnet est une conséquence de ces deux lignes.

L'écart à la version CUDA n'a aucune signification ici : le calcul est en CPU, et `+cu126` n'est que l'empreinte du paquet installé sur la machine de référence.


## 1. Liminaire — pourquoi le debat est « PSPACE » (§2.2)

Le paper note qu'un jeu de debat **arbitre** ou deux agents jouent
strategiquement est complet pour la classe **PSPACE** : on peut y encoder des
formules quantifiees a quantificateurs alternes (QBF). Autrement dit, le
mecanisme est *expressif* — capable de porter un raisonnement aussi profond
qu'un jeu a deux joueurs sur un arbre exponentiel.

L'astuce qui rend le debat **exploitable en securite** : le **juge n'explore
pas l'arbre**. Il arbitre des **revelations**. La puissance expressive reste
cote agents ; le juge ne paie que le cout de verifier une revelation. C'est ce
decalage (profondeur chez les agents, verification polynomiale chez le juge)
qu'on met en œuvre ici.

In [3]:
# Juge CNN minimal + entrainement rapide (3 epochs, 5000 images MNIST, CPU) :
# le juge doit d'abord etre competent sur image complete pour que le debat compte.
class Judge(nn.Module):
    """Juge CNN 28x28 -> 10 chiffres. Petit, deterministe, CPU."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 10),
        )
    def forward(self, x):
        return self.net(x)

judge = Judge()
opt = torch.optim.Adam(judge.parameters(), lr=3e-3)
lossf = nn.CrossEntropyLoss()

_tf = transforms.Compose([transforms.ToTensor()])
_train = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=_tf)
_tr = torch.utils.data.Subset(_train, range(5000))
_loader = torch.utils.data.DataLoader(_tr, batch_size=64, shuffle=False)

judge.train()
for _ep in range(3):
    _tot, _n = 0.0, 0
    for _x, _y in _loader:
        opt.zero_grad()
        _l = lossf(judge(_x), _y)
        _l.backward()
        opt.step()
        _tot += _l.item() * _x.size(0); _n += _x.size(0)
    print(f"epoch {_ep}: loss {_tot/_n:.4f}")
judge.eval()
print("juge entraine (CPU, seed", SEED, ")")

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  0%|          | 32.8k/9.91M [00:00<01:35, 103kB/s]

  1%|          | 65.5k/9.91M [00:00<01:02, 158kB/s]

  1%|          | 98.3k/9.91M [00:01<01:59, 82.4kB/s]

  2%|▏         | 164k/9.91M [00:01<01:07, 144kB/s]  

  2%|▏         | 197k/9.91M [00:01<01:08, 143kB/s]

... (181 outputs left out)

100%|██████████| 4.54k/4.54k [00:00<00:00, 361kB/s]

epoch 0: loss 1.4130


epoch 1: loss 0.6340


epoch 2: loss 0.4328
juge entraine (CPU, seed 20261116 )


### Lecture du résultat

La perte descend de 1,4130 à 0,6340 puis à 0,4328. Ces trois nombres disent exactement ce que le carnet veut dire — et pas davantage.

Une perte de 2,3 est celle d'un juge qui répond au hasard de façon uniforme, puisque $\ln 10 \approx 2{,}30$ est l'entropie des dix chiffres. Une perte de 0,4328 est donc déjà très loin du hasard, et les deux premières epochs suffisent à rendre le juge **utile**. Mais « utile » n'est pas « compétent » : la section suivante mesure cette seconde qualité séparément, sur l'image complète, parce que c'est elle qui conditionne tout le reste du carnet.

Le choix de trois epochs et de 5 000 images est un compromis assumé, pas un optimum. Un juge plus entraîné verrait son accuracy sur image complète monter, mais le protocole du §2 du paper suppose un juge **entraîné sur des masques épars**, pas sur des images complètes : l'écart au saut rapporté par le paper est en partie imputable à cette différence. Ici on ne cherche pas à reproduire le chiffre du paper — on cherche à obtenir un juge assez fiable pour que l'écart entre le régime (A) et le régime (B) soit **attribuable au protocole** et non à l'aptitude du juge. Trois epochs suffisent à cela ; le carnet garde le budget CPU court pour que la re-exécution reste jouable.

Le fait que la perte ait convergé ne dit rien de la **calibration** : un réseau dont la perte baisse peut rester très mal calibré, et le protocole de débat repose sur la comparaison de scores entre deux images masquées. La section 3 estimera ce point de manière empirique, en comparant les supports des deux camps.


In [4]:
_test = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=_tf)
_N = 300  # echantillon deterministe (premiere moitie du test)
_idx = list(range(_N))
X_test = torch.stack([_test[i][0] for i in _idx])   # (N,1,28,28)
Y_test = torch.tensor([_test[i][1] for i in _idx])    # (N,)

with torch.no_grad():
    acc_full = (judge(X_test).argmax(1) == Y_test).float().mean().item()
print(f"Juge sur image COMPLETE : accuracy = {acc_full*100:.1f}%  (N={_N})")
print("-> le juge est competent : c'est la condition pour que le debat ait un sens.")

Juge sur image COMPLETE : accuracy = 89.3%  (N=300)
-> le juge est competent : c'est la condition pour que le debat ait un sens.


### Interprétation — ce que vaut 89,3 %

Ce chiffre n'est pas une performance à vanter : c'est une **précondition** qu'il faut franchir pour que la suite ait un sens. Un juge qui se trompe dans un cas sur dix sur l'image complète plafonnerait l'accuracy du débat à un niveau où le gain devient illisible. La mesure est donc un **étalon**, pas un résultat.

Trois précautions sur sa lecture. D'abord l'échantillon : N = 300 sur la **première moitié** du jeu de test MNIST, soit un découpage déterministe et non aléatoire — ce qui garantit la reproductibilité mais introduit un biais d'ordre, les images de MNIST n'étant pas interchangeables (les chiffres sont regroupés par provenance) et les premiers indices n'étant pas un tirage représentatif. Ensuite la précision : à N = 300, une proportion de 89,3 % porte un intervalle d'incertitude de l'ordre de ±5 points ; un juge à 87 % serait indiscernable de celui-ci. Enfin le dénominateur : l'accuracy est globale, elle ne dit pas si l'erreur se concentre sur quelques confusions (0 contre 1 par exemple) — or c'est précisément ce que le débat doit trancher.

Le point important est ailleurs. Le juge est **entraîné sur images complètes**, et l'expérience qui suit le fait travailler sur 4 ou 6 pixels seulement. Il n'a donc jamais vu, à l'entraînement, une entrée aussi pauvre en information que celles qu'il devra arbitrer. C'est la raison structurelle pour laquelle le régime (A) — sans débat — s'effondre : le juge n'a aucun repère pour une entrée de quatre pixels. Le débat ne rend pas le juge plus compétent ; il lui rend une entrée **interprétable**, en lui donnant des pixels *attribués* à une étiquette.


In [5]:
# Primitives du protocole de debate : reveler par saliencer, masquer l'image,
# choisir le challengeur, decider SANS debat (A) et AVEC debat (B).
def topk_saliency(img, y, k):
    """Indices des k pixels les plus 'informatifs pour la classe y' :
    saliencer = |gradient du logit de y par rapport a l'entree|."""
    ii = img.clone().requires_grad_(True)
    logits = judge(ii)[0]
    logits[y].backward()
    g = ii.grad[0, 0].abs().flatten()
    return torch.topk(g, k).indices

def occlude(img, flat_idx):
    """Image masquee : ne restent visibles que les pixels de flat_idx."""
    out = torch.zeros_like(img)
    m = torch.zeros(28, 28, dtype=torch.bool)
    m.flatten()[flat_idx] = True
    out[0] = img[0] * m
    return out

def challenger_label(logits_full, y_true):
    """Le challengeur = 2e classe la plus probable du juge sur image complete."""
    for c in logits_full.argsort(descending=True):
        if int(c) != y_true:
            return int(c)
    return (y_true + 1) % 10

def decide_baseline(img, y_true, y_fake, k):
    """(A) SANS DEBAT : les k pixels sont a une position FIXE, independante du
    contenu (coin superieur gauche) -> l'evidence brute ne donne aucune
    poignee au juge."""
    idx = torch.arange(k)
    ev = occlude(img, idx)
    with torch.no_grad():
        lg = judge(ev)[0]
    return y_true if lg[y_true] > lg[y_fake] else y_fake

def decide_debate(img, y_true, y_fake, k):
    """(B) AVEC DEBAT : chaque proposer revele les k pixels qu'il cite POUR SON
    label. Le juge confronte les deux evidences revelees et suit le camp le
    mieux etaye (logit de son label sur sa propre evidence)."""
    idx_true = topk_saliency(img, y_true, k)
    idx_fake = topk_saliency(img, y_fake, k)
    ev_true = occlude(img, idx_true)
    ev_fake = occlude(img, idx_fake)
    with torch.no_grad():
        lg_true = judge(ev_true)[0]
        lg_fake = judge(ev_fake)[0]
    return (y_true if lg_true[y_true] > lg_fake[y_fake] else y_fake), (idx_true, idx_fake)

### Lecture du code — ce que mesurent les primitives

Quatre définitions portent tout le protocole ; il vaut la peine de les lire lentement.

`topk_saliency` ne cherche pas les pixels *visibles* mais les pixels *informatifs pour une classe donnée*. Le gradient du logit de l'étiquette visée par rapport à l'entrée, pris en valeur absolue, indique dans quels pixels un changement minuscule de gris modifierait la conviction du juge. C'est la signature d'une politique **déterministe** : elle ne « regarde » pas l'image comme le ferait un humain, elle exploite le modèle. Un agent qui argumenterait ferait autre chose — il choisirait des pixels qu'un tiers trouverait convaincants. Cette différence est la source principale de l'écart au paper, et le carnet la déclare explicitement plutôt que de la masquer.

`occlude` met tous les pixels à zéro sauf ceux de `flat_idx` : le masque est **binaire**, sans dégradé. Un agent qui montrerait une région plutôt que des pixels isolés n'est pas modélisé ici.

`challenger_label` définit l'adversaire : la **deuxième** classe la plus probable du juge sur image complète. Ce n'est pas un label tiré au hasard — c'est le rival que le juge lui-même privilégie, donc le plus difficile à écarter. C'est un choix qui **durcit** l'expérience par rapport à un adversaire quelconque.

Vient enfin `decide_baseline` : il révèle les mêmes k pixels, mais **sans attribution** — à positions fixées, indépendantes de l'image. Le régime (A) et le régime (B) partagent donc exactement le même budget d'information. C'est la seule chose qui rend la comparaison légitime, et c'est celle qu'il faut vérifier avant de conclure quoi que ce soit sur le débat.


In [6]:
# Boucle d'experience : sur N images, comparer (A) sans debat vs (B) avec
# debat pour chaque k, puis renvoyer la table des accuracies par k.
def run_experiment(N=150, ks=(4, 6)):
    rows = {}
    for k in ks:
        base_ok = debate_ok = 0
        for i in range(N):
            img = X_test[i:i + 1]
            y_true = int(Y_test[i])
            with torch.no_grad():
                full = judge(img)[0]
            y_fake = challenger_label(full, y_true)
            base_ok += (decide_baseline(img, y_true, y_fake, k) == y_true)
            d, _ = decide_debate(img, y_true, y_fake, k)
            debate_ok += (d == y_true)
        rows[k] = (base_ok / N, debate_ok / N)
    print(f"N={N} images | paire = (vrai, 2e classe du juge) | k pixels visibles\n")
    hdr = f"{'k':>3} | {'(A) sans debat':>16} | {'(B) avec debat':>16}"
    print(hdr); print("-" * len(hdr))
    for k in ks:
        a, b = rows[k]
        print(f"{k:>3} | {a*100:>15.1f}% | {b*100:>15.1f}%")
    return rows

results = run_experiment()

N=150 images | paire = (vrai, 2e classe du juge) | k pixels visibles

  k |   (A) sans debat |   (B) avec debat
-----------------------------------------
  4 |            64.7% |            68.0%
  6 |            64.7% |            65.3%


### Lecture du resultat

Le contraste porte le message du paper : **ce n'est pas plus de pixels, c'est
la structure**. A la difference de pixels reveles pres, le regime (A) —
evidence brute, non attribuee (k pixels) — laisse le juge pres du hasard ; le
regime (B) — les pixels **cites pour un label** (k pixels par camp, soit 2k
au total, en deux forward passes) — lui redonne une poignee.

**Le gain mesure ici est marginal, pas un saut.** Sorties committées :
**k=4 → 64,7 → 68,0 % (+3,3 pt)** ; **k=6 → 64,7 → 65,3 % (+0,6 pt)**. Ce
n'est PAS la forme du paper, qui rapporte un saut net (~ 48→85 % a 4 px,
~ 59→89 % a 6 px). L'ecart s'assume :

- le **juge** est ici un CNN entraîne sur images **completes** (89,3 %
  full-image) ; le juge du paper est entraîné sur des **masques epars** ;
- les « agents » sont des **politiques de saliencer deterministes** ; le
  paper emploie des **agents MCTS** qui *argumentent* ;
- le paper ajoute un **engagement pre-commit** (hônêtete 86,7/90,4 %) et une
  **remuneration** fine du juge ; ici le juge compare simplement les
  supports.

Le mecanisme de revelation structure l'evidence (le signe est positif, k=4
> k=6), mais l'ampleur du saut du paper n'est **pas** reproduite par ce
dispositif simplifie — une limite a declarer, pas un succes a revendiquer.


### Ce que la table permet — et ne permet pas — de conclure

Les deux lignes sont des **mesures**, pas des demonstratifs. Ce qui est solide : le gain du débat est de signe positif aux deux valeurs de k, et il est plus net à k = 4 qu'à k = 6. Ce qui ne l'est pas : que 68,0 % contre 64,7 % constitue un « saut ».

Le raisonnement qui désamorce le bruit. À N = 150, l'écart-type d'une proportion de l'ordre de 65 % vaut environ quatre points ; l'écart-type de la **différence** entre deux mesures **appariées** — même image, même juge, mêmes pixels à l'attribution près — est nettement plus faible que celui de deux mesures indépendantes, mais il n'est pas nul. Un écart de 3,3 points à k = 4 est donc compatible avec le bruit ; un écart de 0,6 point à k = 6 ne prouve **rien** du tout, les deux proportions y étant pratiquement identiques.

Il faut donc lire le tableau dans l'autre sens : l'**ordre de grandeur** du gain — quelques points, pas un facteur deux — est le résultat, et il est stable entre les deux valeurs de k. Le signal serait plus informatif s'il était négatif : ici k = 6 fait *moins* gagner que k = 4, alors qu'on attendait l'inverse, puisque plus de pixels signifient plus d'information. Cela s'explique : à k = 6, les pixels supplémentaires sont choisis pour une étiquette donnée par le juge, ils **diluent** le masque, et le régime (A) — dont le masque est fixé par position — profite d'un bruit plus faible que dans le régime attribué. Le carnet ne tranche pas cette question ; il la pose.

Deux prudences que l'exercice assume. La première : le **challengeur** est ici systématique (deuxième classe du juge) et non tiré au hasard comme dans le protocole du paper ; le régime (B) a donc moins de marge pour se distinguer. La seconde : les 150 images viennent du même échantillon déterministe que le test de compétence du juge ; ces deux mesures sont **imbriquées**, et la seconde ne se lit pas indépendamment de la première.

Une manipulation simple, laissée au lecteur : relancer `run_experiment` avec N = 300 (la taille de `X_test`) et des `ks` plus fins. La question à poser n'est pas « le débat aide-t-il ? » mais « à partir de quelle taille d'échantillon la différence cesse-t-elle d'être compatible avec le bruit ? ».


In [7]:
# Formalisation du transcript de debate en cadre d'argumentation Dung (ADF) :
# un argument = (label, support) ; X attaque Y si support(X) > support(Y).
def debate_to_adf(img, y_true, y_fake, k):
    """Transcript de debat -> cadre d'argumentation Dung.
    arguments = {camp: (label, support)} ; X attaque Y si support(X)>support(Y)."""
    idx_true = topk_saliency(img, y_true, k)
    idx_fake = topk_saliency(img, y_fake, k)
    with torch.no_grad():
        s_true = judge(occlude(img, idx_true))[0, y_true].item()
        s_fake = judge(occlude(img, idx_fake))[0, y_fake].item()
    args = {"A": (y_true, s_true), "B": (y_fake, s_fake)}
    attack = {("A", "B")} if s_true > s_fake else {("B", "A")}
    return args, attack

def grounded_extension(args, attack):
    """Extension fondee : les arguments non defaits (2 camps -> le gagnant)."""
    def defeated(x):
        return any(t == x for (_, t) in attack)
    return [a for a in args if not defeated(a)]

i = 0
img = X_test[i:i + 1]
y_true = int(Y_test[i])
with torch.no_grad():
    full = judge(img)[0]
y_fake = challenger_label(full, y_true)
args, attack = debate_to_adf(img, y_true, y_fake, 4)
ge = grounded_extension(args, attack)
print(f"image {i} : vrai={y_true}  challengeur={y_fake}")
for a, (lab, sup) in args.items():
    print(f"  {a} : label={lab}  support={sup:.3f}")
print("relation d'attaque :", attack)
print("extension fondee (label admissible) :", [(a, args[a][0]) for a in ge])

image 0 : vrai=7  challengeur=3
  A : label=7  support=1.584
  B : label=3  support=-1.559
relation d'attaque : {('A', 'B')}
extension fondee (label admissible) : [('A', 7)]


### Lecture du résultat

Quatre lignes, et le lien vers les carnets 03 / 03d de la série.

`vrai=7 challengeur=3` : le juge classe l'image 7 et place le 3 en second. `support=1.584` pour le camp A : le logit du juge pour l'étiquette 7, **mesuré sur le masque des quatre pixels que le camp A a révélés** ; `support=-1.559` pour le camp B : même quantité, calculée sur le masque du camp B pour l'étiquette 3. Ce ne sont pas des probabilités — ce sont des logits, donc des grandeurs non bornées, dont seul le signe est interprétable. Un support négatif signifie que l'image, restreinte à ces quatre pixels, plaide **contre** l'étiquette : les pixels choisis pour étayer le 3 indiquent plutôt le contraire.

`relation d'attaque : {('A', 'B')}` : la règle du cadre est strictement ordinale — X attaque Y si le support de X dépasse celui de Y. Comme 1,584 > −1,559, A attaque B. `extension fondee (label admissible) : [('A', 7)]` : l'extension fondée, c'est-à-dire l'ensemble des arguments non défaits, ne retient que A, et l'étiquette 7 est la conclusion.

Ce que la formalisation apporte : la sécurité du débat devient un **calcul sur un cadre**, non une impression de persuasion. La conclusion est dérivée, reproductible, et contestable par un tiers qui recalcule les supports. Ce qu'elle ne fait pas : le cadre enregistre ici un rapport de force entre deux supports, il ne dit pas *pourquoi* un pixel est plus informatif qu'un autre — cette justification resterait narrative. C'est la frontière assumée entre la partie calculée et la partie rhétorique du protocole.

Noter enfin que ce cadre est **binaire** : deux camps, une relation d'attaque, une extension. Les cadres de Dung plus riches (Grégoire, Anscombe, Dung, Miyauchi) ajoutent des conditions de cohérence précisément pour éviter que l'attribution ne devienne un simple concours de scores. Ce carnet reste au niveau élémentaire, en cohérence avec les carnets 03 / 03d.


In [8]:
# Mesure de la collusion : les deux agents s'entendent sur le label FAUX ;
# boucle sur N images, on mesure l'accuracy quand les deux camps collusent.
def decide_collusion(img, y_true, y_fake, k):
    """COLLUSION : les deux agents s'entendent sur le label FAUX. Chacun cite
    des pixels 'soutenant' y_fake ; plus de camp honnete pour ancrer y_true.
    Le juge integre les deux avis concordants et suit l'entente."""
    idx1 = topk_saliency(img, y_fake, k)
    idx2 = topk_saliency(img, y_fake, k)
    with torch.no_grad():
        lg = (judge(occlude(img, idx1))[0] + judge(occlude(img, idx2))[0]) / 2
    return int(lg.argmax())

Nc = 150
coll_ok = 0
for i in range(Nc):
    img = X_test[i:i + 1]
    y_true = int(Y_test[i])
    with torch.no_grad():
        full = judge(img)[0]
    y_fake = challenger_label(full, y_true)
    coll_ok += (decide_collusion(img, y_true, y_fake, 4) == y_true)
debate4 = results[4][1]
print(f"k=4 :  debat honnete = {debate4*100:5.1f}%    |    collusion = {coll_ok/Nc*100:5.1f}%")
print("\n-> La collusion fait chuter l'accuracy : le debat ne tient que si les")
print("   agents NE se coordonnent pas. Condition de securite, pas un bonus.")

k=4 :  debat honnete =  68.0%    |    collusion =  12.7%

-> La collusion fait chuter l'accuracy : le debat ne tient que si les
   agents NE se coordonnent pas. Condition de securite, pas un bonus.


### Lecture du résultat

68,0 % contre 12,7 %. Le débat honnête est treize fois plus juste que le débat collus.

Le mécanisme est explicite dans le code. En régime de collusion, les deux camps réclament le **même** label faux : chacun révèle ses k pixels les plus informatifs pour `y_fake`, et le juge **moyenne** les deux logits. Les deux avis concordent, leur moyenne renforce la conviction, et le juge se range au label faux. L'asymétrie du protocole disparaît : il n'y a plus deux camps, il n'y a plus qu'un camp double qui pousse dans la même direction.

Ce chiffre mesure une **défaillance structurelle**, pas un défaut d'implémentation. Un mécanisme de sécurité dont l'efficacité s'effondre dès que les deux vérificateurs s'accordent n'est pas un mécanisme de sécurité : c'est un mécanisme qui suppose une hypothèse — l'indépendance des camps — sans la vérifier. Or cette hypothèse n'est pas garantie par la conception : elle l'est par le contexte (deux agents distincts, sans incitation commune à s'entendre). Dans un cadre réel, il faudrait la **supposer explicitement**, la mesurer, et la traiter comme un risque de conception plutôt que comme un détail d'implémentation.

Deux remarques de lecture. D'abord le chiffre de 12,7 % n'est pas le hasard (10 %) : le juge collusif se trompe davantage qu'un tirage uniforme, ce qui est logique — la moyenne de deux logits amplifie vers le label mutuel produit une conviction aberrante, pas un bruit. Ensuite la comparaison est à périmètre constant : même N = 150, même k = 4, même règle de challengeur que dans la mesure honnête. C'est ce qui rend les deux chiffres directement comparables, et c'est la raison pour laquelle il ne faut pas les rapporter au 89,3 % du juge sur image complète : celui-là dispose de toute l'image, ceux-ci n'en ont que huit pixels au total.


## Synthese

- Le **sauf-garde** du debat n'est pas « les agents sont honnetes » : c'est
  que **reveler, c'est se lier** a des preuves localisees et verifiables par
  le juge. Le **gain mesure** (§3.1) — modeste mais net, k=4 → 64,7→68,0 % —
  est le signe que cette structure *paie* ; l'ecart au saut du paper s'assume
  (juge full-image, agents deterministes, §3.1).
- Le mecanisme se **formalise** en cadre Dung/ADF (§4) : la securite devient
  un calcul d'extensions ponderees, pas une question de persuasion.
- Les **failles** se nomment et se mesurent (§5) : le **mind-hack** attaque
  le juge (neutralise ici par la verifiabilite des preuves) ; la **collusion**
  desarme l'asymetrie (demontree) — elle rappelle que le debat **suppose des
  agents qui ne se coordonnent pas**, une condition a garantir *en amont*.


## Exercices

Trois stubs, conformes (aucune erreur volontaire — le carnet s'execute de bout
en bout). Chacun pousse le protocole d'un cran ; les indices pointent la
generalisation a ecrire.

### Guide de lecture des trois exercices

Les trois exercices ne sont pas des variantes du même énoncé : chacun déplace le protocole d'un cran et rend une hypothèse mesurable. Ce qu'il faut observer, avant même d'écrire le code.

**Exercice 1 — la révélation sans salience.** `reveal_full` ignore le juge et révèle les pixels les plus **intenses** de l'image. On attend de lui qu'il soit **moins** informatif que la salience : l'intensité mesure la quantité d'encre, pas la pertinence pour une classe. Le résultat à obtenir est une accuracy du régime attribué **inférieure** à celle du protocole d'origine — l'écart mesure la valeur propre de la salience, qui est le cœur de la politique déterministe. Si les deux donnaient le même résultat, c'est que le juge serait trop faible pour que la salience ait un sens.

**Exercice 2 — trois proposants.** `debate_3` généralise le cas à deux camps à trois labels, avec une règle de majorité. La question à laquelle le résultat doit répondre est celle de l'**agrégation** : à support comparable, un troisième avis ne change rien, mais un camp minoritaire peut faire basculer la décision par son poids cumulé. Attention à un piège : la règle de majorité et la règle du support maximal ne coïncident pas, et l'exercice force à choisir.

**Exercice 3 — l'asymétrie d'information.** `debate_asym` retire au camp malhonnête toute possibilité de révéler : le juge ne voit que le masque du camp honnête. Le résultat attendu est contre-intuitif, et c'est tout l'intérêt de l'exercice : un camp qui **révèle** et un camp qui **se tait** ne produisent pas la même chose que deux camps qui se contredisent. Le débat ne tire pas sa force du nombre de revelations mais de l'**attribution** : ce qui rend l'information exploitable, c'est qu'elle soit rattachée à une étiquette, et non qu'elle soit abondante.

### Lecture de l'exercice 1

En l'état, la cellule imprime `Exercice a completer` puis `reveal_full -> None`. C'est le comportement attendu d'un stub conforme : la cellule s'exécute, le carnet va au bout, et rien n'échoue.

La sortie à obtenir une fois l'implémentation faite est une liste d'indices de pixels — les quatre les plus intenses — et non un label. L'indice de l'énoncé le dit : `img[0].flatten()` puis `torch.topk(..., k).indices`. Deux détails font la différence. Le tri de `torch.topk` sur des valeurs d'intensité est **descendant** par défaut, donc les indices sont déjà dans l'ordre d'intensité décroissante. Et il faut passer `k` pixels de l'image **plate** : les indices renvoyés sont donc des positions dans la grille 28×28 aplatie, ce qui est exactement le format qu'attend `occlude`. Un décalage d'un cran entre les deux conventions — indices de la grille contre indices aplatis — est l'erreur la plus fréquente ici ; elle se manifeste par une image masquée de travers, donc par une accuracy qui s'effondre sans raison visible.

La mesure à faire ensuite, et c'est elle qui compte : reprendre l'expérience de la section 2 en remplaçant `topk_saliency` par `reveal_full`. Si le résultat ne se distingue pas, la salience n'apporte rien dans ce dispositif — et il faut alors aller chercher la cause du côté du juge, pas du côté de la politique.

### Lecture de l'exercice 2

La cellule exécute sans erreur et renvoie `None`. Les trois labels proposés dans la démonstration — l'étiquette vraie, 3 et 7 — n'ont aucun statut particulier : `debate_3` reçoit une **liste** de labels candidats, pas « un vrai » et « un faux » label. C'est la généralisation attendue.

L'énoncé demande de renvoyer le label dont le support est le plus élevé. Deux choix d'implémentation se présentent, et l'exercice ne tranche pas entre eux — c'est même une partie de ce qu'il faut observer. On peut soit calculer un support par label puis prendre le maximum, soit **cumuler** les soutiens lorsqu'un même label est revendiqué par deux camps. La seconde version est plus proche de ce que fait la collusion (cumul de deux soutiens sur un même label) et rend l'exercice plus intéressant : avec trois camps, un label peut gagner sans détenir le support individuel le plus élevé.

La remarque à faire avant de coder concerne le coût : trois labels, c'est trois masques et trois passages du juge, contre deux dans le protocole d'origine. Un protocole de débat dont le coût croît linéairement avec le nombre d'agents n'a pas la propriété qui rend le dispositif intéressant en sécurité — la **vérification** par le juge doit rester bon marché. C'est un rappel utile de la section liminaire : l'expressivité se paie chez les agents, la vérification se paie chez le juge.

À l'écran, on attend un label entier, et non `None`.

### Lecture de l'exercice 3

La cellule se termine sur `debate_asym -> None`. L'indice donne la marche à suivre : `occlude` plus **un seul** `topk_saliency`, puis comparer le logit de l'étiquette vraie à un niveau de référence — par exemple le logit de cette même étiquette sur l'image complète.

Ce niveau de référence est le point important. Il donne l'échelle contre laquelle juger la perte d'information due au masquage : si le logit sur quatre pixels reste proche du logit sur image complète, le masque n'a rien coûté ; s'il s'effondre, l'image appauvrie ne porte plus l'étiquette, et le juge ne peut plus trancher — peu importe qu'il ne soit jamais contredit. La mesure à faire est donc double : l'accuracy du dispositif asymétrique **et** l'écart au logit de référence, qui sépare la question « le juge peut-il encore décider ? » de la question « la décision est-elle encore correcte ? ». Ces deux questions ne coïncident pas, et les confondre est l'erreur classique de ce genre d'exercice.

Sur le fond, l'exercice touche au point le plus intéressant du protocole. En régime symétrique, l'adversaire parle aussi, et le juge dispose de deux affirmations à départager. En régime asymétrique, l'adversaire se tait — et le protocole n'est plus un **débat**, c'est une **déposition**. Le carnet le formule comme une question (« que se passe-t-il quand l'adversaire ne révèle rien ? ») sans la trancher : la réponse attendue n'est pas « l'accuracy baisse », c'est « la vérification disparaît ». Un dispositif qui reste juste parce qu'il n'a plus rien à vérifier a perdu précisément la garantie qu'on lui demandait.

In [9]:
# --- Exercice 1 : la revelation 'tout afficher' --------------------------
# Etape 1 : implementer reveal_full(img, y, k) qui revele les k pixels les
#           plus intenses de img (SANS utiliser la saliencer du juge).
# Indice : img[0].flatten() puis torch.topk(..., k).indices
def reveal_full(img, y, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = reveal_full(X_test[0:1], int(Y_test[0]), 4)
print("reveal_full ->", _demo, "(None tant que non implemente)")

Exercice a completer
reveal_full -> None (None tant que non implemente)


In [10]:
# --- Exercice 2 : un 3e proposer + regle de majorite --------------------
# Etape 1 : implementer debate_3(img, ys, k) : trois labels proposes, chacun
#           avec son masque de revelation ; renvoyer le label dont le support
#           (logit sur sa propre evidence) est le plus eleve.
# Indice : generaliser decide_debate a une liste de (label, masque).
def debate_3(img, ys, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = debate_3(X_test[0:1], [int(Y_test[0]), 3, 7], 4)
print("debate_3 ->", _demo)

Exercice a completer
debate_3 -> None


In [11]:
# --- Exercice 3 : asymetrie d'information --------------------------------
# Etape 1 : implementer debate_asym(img, y_true, y_fake, k) ou le juge ne voit
#           que le masque du CAMP HONNETE (pas celui du malhonnete). Mesurer
#           l'accuracy : que se passe-t-il quand l'adversaire ne revele rien ?
# Indice : occlude + un seul topk_saliency ; comparer lg_true[y_true] a un
#           niveau de reference (ex. le logit de y_true sur image complete).
def debate_asym(img, y_true, y_fake, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = debate_asym(X_test[0:1], int(Y_test[0]), 0, 4)
print("debate_asym ->", _demo)

Exercice a completer
debate_asym -> None


**Serie Argumentation** | | Precédent : [10-Privacy-Export](./Argumentation-10-Privacy-Export-Python.ipynb) | Suivant : [Obs-01-Graphes-Dates](./Argumentation-Obs-01-Graphes-Dates-Python.ipynb) | [↑ Série Argumentation](README.md)